Calculates the performance characteristics of a trapezoidal, 3D wing based on input parameters.

Uses a custom spanwise-loading model for the trapezoidal planform instead of assuming an elliptical lift distribution.

In [176]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [177]:
from pathlib import Path
import numpy as np
import af_2d

# Use a workspace-relative path so the notebook runs from the repo root
file_path = Path.cwd() / "airfoil-performance" / "s7055.csv"
if not file_path.exists():
    file_path = Path("airfoil-performance") / "s7055.csv"
if not file_path.exists():
    raise FileNotFoundError(f"Airfoil data not found: {file_path}")

rho = 1.225  # kg/m^3
V = 10.0     # m/s
alpha_deg = 1
alpha = np.deg2rad(alpha_deg)

b = 5.0      # wingspan, m
c_r = 0.2    # root chord, m
taper_ratio = 1.0  # c_t / c_r
n_span = 400

# 2D airfoil coefficients from the polar data
cl_0 = af_2d.get_C_l_0(str(file_path))
cd_0 = af_2d.get_C_d_0(str(file_path))
a0 = af_2d.get_C_l_vs_alpha(str(file_path))    # 1/rad
alpha_l_0 = af_2d.get_alpha_l_0(str(file_path))  # radians

# Trapezoidal geometry: actual chord distribution over one half-span
# c(y) = c_root * (1 - (1 - taper_ratio) * (2|y|/b))
y = np.linspace(0.0, b / 2.0, n_span)
eta = 2.0 * y / b
c = c_r * (1.0 - (1.0 - taper_ratio) * eta)

# Local section lift coefficient based on the actual airfoil data,
# adjusted for the local angle of attack relative to zero-lift.
alpha_sec = alpha - alpha_l_0
cl_local = cl_0 + a0 * alpha_sec

# Non-elliptic spanwise loading for a tapered trapezoidal wing.
# The factor goes to zero at the tips while preserving the actual section lift.
span_factor = 1.0 - eta**2
cl_dist = cl_local * span_factor

# Lift and drag from the actual spanwise loading over the planform
q = 0.5 * rho * V**2
lift_per_span = q * c * cl_dist
S = b * 0.5 * c_r * (1.0 + taper_ratio)
C_L = (2.0 / S) * 2.0 * np.trapezoid(c * cl_dist, y)

AR = b**2 / S
lift_efficiency = 0.88
cd_i_local = cl_dist**2 / (np.pi * AR * lift_efficiency)
C_D_i = (2.0 / S) * 2.0 * np.trapezoid(c * cd_i_local, y)
C_D = cd_0 + C_D_i

print("Spanwise lift distribution (half-span):")
for yi, li, ci, cli in zip(y[:8], lift_per_span[:8], c[:8], cl_dist[:8]):
    print(f"y={yi:>5.3f} m  c={ci:>6.4f} m  cl={cli:>6.4f}  l'={li:>8.5f} N/m")
print("-" * 72)
print(f"AOA = {alpha_deg:.1f} degrees")
print(f"AR = {AR:.4f}")
print(f"cl_0 = {cl_0:.4f}")
print(f"cd_0 = {cd_0:.4f}")
print(f"alpha_l_0 = {np.rad2deg(alpha_l_0):.4f} deg")
print(f"a0 = {a0:.4f} /rad")
print(f"C_L = {C_L:.4f}")
print(f"C_D_i = {C_D_i:.4f}")
print(f"C_D_total = {C_D:.4f}")
print(f"L/D = {C_L / C_D:.4f}")
print(f"Lift distribution integration check: total lift per half-span = {np.trapezoid(lift_per_span, y):.3f} N")


Spanwise lift distribution (half-span):
y=0.000 m  c=0.2000 m  cl=0.7407  l'= 9.07336 N/m
y=0.006 m  c=0.2000 m  cl=0.7407  l'= 9.07330 N/m
y=0.013 m  c=0.2000 m  cl=0.7407  l'= 9.07313 N/m
y=0.019 m  c=0.2000 m  cl=0.7406  l'= 9.07284 N/m
y=0.025 m  c=0.2000 m  cl=0.7406  l'= 9.07244 N/m
y=0.031 m  c=0.2000 m  cl=0.7406  l'= 9.07193 N/m
y=0.038 m  c=0.2000 m  cl=0.7405  l'= 9.07130 N/m
y=0.044 m  c=0.2000 m  cl=0.7405  l'= 9.07056 N/m
------------------------------------------------------------------------
AOA = 1.0 degrees
AR = 25.0000
cl_0 = 0.3952
cd_0 = 0.0181
alpha_l_0 = -2.4274 deg
a0 = 5.7754 /rad
C_L = 0.9876
C_D_i = 0.0085
C_D_total = 0.0266
L/D = 37.1872
Lift distribution integration check: total lift per half-span = 15.122 N
